# Notebook 06 — CheMeleon foundation model for four CYP direct-inhibition endpoints

**Experiments:** (1) **frozen CheMeleon graph embeddings → LightGBM** and a regularized linear reference, (2) **Chemprop D-MPNN trained from scratch** with four simultaneous targets, and (3) **CheMeleon-initialized multi-task Chemprop D-MPNN fine-tuning**.

- Uses the **same fixed compound-level scaffold folds** as Notebook 05. Optionally reads `fold_manifest.csv` from its results ZIP; otherwise deterministically regenerates the manifest and asserts no scaffold overlap.
- No experimental measurement for a validation molecule enters any training head. The four pIC50 targets use masked multi-task losses for missing labels. Scaling is fitted **only on inner-training data**.
- The inner validation partition is used for checkpoint selection; outer validation labels are never used for training or early stopping. By default the final outer prediction uses the checkpoint trained without the inner-validation compounds; this is deliberately conservative for a clean comparison.
- **Default is a 1-fold, 6-epoch Colab GPU smoke run** (`FAST_MODE=True`). To obtain a complete out-of-fold benchmark, set `FAST_MODE=False`, which runs **all five outer folds** with longer training. A single-fold result must not be compared as if it were five-fold OOF.
- Downloads CheMeleon pretrained message-passing weights from the authors' [Zenodo archive](https://zenodo.org/records/15460715); citation required for publication. Model architecture follows [Chemprop's official CheMeleon tutorial](https://chemprop.readthedocs.io/en/latest/chemeleon_foundation_finetuning.html). Installation and checkpoint download need internet access.

**Scoring:** reports MAE, R², rank correlations, and transparent confidence-interval excess distance. **The latter is not the exact official leaderboard MA-ST-RAE.** Do not report it as such.

Use **Runtime → Change runtime type → T4 GPU**, then Runtime → Run all. This notebook does not require importing project-local source files.

## 0 · Install Chemprop, RDKit, and analysis libraries

In [ ]:
%pip -q install "chemprop>=2.2.0,<2.4" "rdkit>=2023.9" "scikit-learn>=1.6" "lightgbm>=4.3" "lightning>=2.4" pandas scipy matplotlib

## 1 · Configuration and data

In [ ]:
import os, io, sys, json, platform, hashlib, zipfile, warnings, time, random
from pathlib import Path
from urllib.request import urlopen, urlretrieve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
from rdkit import Chem, rdBase
from rdkit.Chem.Scaffolds import MurckoScaffold
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
import chemprop
from chemprop import featurizers, nn, data, models

SEED=2026
N_SPLITS=5
FAST_MODE=True  # Set False for all 5 folds and longer training; use a GPU.
RUN_FROZEN=True
RUN_SCRATCH=True
RUN_CHEMELEON_FINETUNE=True
DATA_OVERRIDE=None    # Optional local CSV for offline testing
NOTEBOOK05_ZIP_PATH=None   # Optional path to cyp_notebook_05_results.zip uploaded in Colab Files
CACHE_DIR=Path('cyp_chemeleon_cache')
OUT=Path('cyp_notebook_06_results')
CACHE_DIR.mkdir(exist_ok=True);OUT.mkdir(exist_ok=True)
FOLDS_TO_RUN=[0] if FAST_MODE else list(range(N_SPLITS))
MAX_EPOCHS=6 if FAST_MODE else 25
BATCH_SIZE=48 if FAST_MODE else 64
INNER_VAL_FRACTION=0.15
ISOFORMS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DATA_URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/cyp-challenge-TRAIN_inhibition.csv'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); pl.seed_everything(SEED,workers=True)
print('Py',platform.python_version(),'Torch',torch.__version__,'Chemprop',getattr(chemprop,'__version__','unavailable'),'RDKit',rdBase.rdkitVersion)
print('CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available() and (RUN_SCRATCH or RUN_CHEMELEON_FINETUNE):
    print('WARNING: GPU is strongly recommended for graph training. Enable T4 GPU in Colab.')
if DATA_OVERRIDE is None:
    with urlopen(DATA_URL,timeout=120) as response: content=response.read()
else:
    content=Path(DATA_OVERRIDE).read_bytes()
data_sha=hashlib.sha256(content).hexdigest()
df=pd.read_csv(io.BytesIO(content))
print('Read data:',df.shape,'SHA256:',data_sha)
assert 'SMILES' in df, 'Expected original SMILES column'
for iso in ISOFORMS:assert f'{iso}_pIC50_direct_inhibition' in df, iso

## 2 · Construct/verify **identical** outer folds across all CYPs

In [ ]:
entry=[]
for orig_row,s in enumerate(df['SMILES']):
    mol=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    if mol is None:continue
    key=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
    scaff=MurckoScaffold.GetScaffoldForMol(mol)
    grp=Chem.MolToSmiles(scaff,canonical=True,isomericSmiles=False) if scaff.GetNumAtoms() else 'ACYCLIC_'+key
    entry.append((orig_row,key,grp,mol))
clean=df.iloc[[r[0] for r in entry]].copy().reset_index(drop=True)
clean['canonical_smiles']=[r[1] for r in entry]
clean['scaffold_group']=[r[2] for r in entry]
assert not clean.canonical_smiles.duplicated().any(),'Resolve duplicate molecular identities before CV'
mols=[r[3] for r in entry]
folds=np.full(len(clean),-1,dtype=int)
for fold,(_,vi) in enumerate(GroupKFold(n_splits=N_SPLITS).split(clean,groups=clean.scaffold_group)):
    folds[vi]=fold
assert (folds>=0).all()
if NOTEBOOK05_ZIP_PATH:
    with zipfile.ZipFile(NOTEBOOK05_ZIP_PATH) as archive:
        filepaths=[x for x in archive.namelist() if x.endswith('/fold_manifest.csv')]
        assert len(filepaths)==1,'Expected a unique fold_manifest.csv in Notebook 05 ZIP'
        source_manifest=pd.read_csv(io.BytesIO(archive.read(filepaths[0])))
        meta_paths=[x for x in archive.namelist() if x.endswith('/metadata.json')]
        if meta_paths:
            source_meta=json.loads(archive.read(meta_paths[0]).decode())
            assert source_meta['dataset_sha256']==data_sha,'Dataset revision differs from Notebook 05'
    assert not source_manifest.canonical_smiles.duplicated().any()
    keyed=source_manifest.set_index('canonical_smiles')['fold']
    assert set(keyed.index)==set(clean.canonical_smiles),'Structure list mismatch vs Notebook 05'
    imported=np.array([int(keyed[s]) for s in clean.canonical_smiles])
    assert np.array_equal(imported,folds),'Fold mismatch! Check Notebook 05 code/version.'
    print('Verified imported Notebook 05 fold manifest and source SHA256.')
else:
    print('Regenerated Notebook 05 split deterministically. To verify independently, provide NOTEBOOK05_ZIP_PATH.')
for fold in range(N_SPLITS):
    assert not (set(clean.loc[folds==fold,'scaffold_group']) & set(clean.loc[folds!=fold,'scaffold_group']))
clean['fold']=folds
clean[['canonical_smiles','scaffold_group','fold']].to_csv(OUT/'fold_manifest.csv',index=False)
Y=clean[[f'{iso}_pIC50_direct_inhibition' for iso in ISOFORMS]].apply(pd.to_numeric,errors='coerce').to_numpy(float)
print('Data shape',Y.shape,'labeled counts',np.isfinite(Y).sum(axis=0),'fold sizes',np.bincount(folds))

## 3 · Load CheMeleon checkpoint (authors’ released message-passing weights)

In [ ]:
CHEMELEON_URL='https://zenodo.org/records/15460715/files/chemeleon_mp.pt'
CHEMELEON_PATH=CACHE_DIR/'chemeleon_mp.pt'
if RUN_FROZEN or RUN_CHEMELEON_FINETUNE:
    if not CHEMELEON_PATH.exists():
        print('Downloading CheMeleon message-passing checkpoint from Zenodo...')
        urlretrieve(CHEMELEON_URL,CHEMELEON_PATH)
    print('Checkpoint size MB:',round(CHEMELEON_PATH.stat().st_size/1e6,2),
          'SHA256:',hashlib.sha256(CHEMELEON_PATH.read_bytes()).hexdigest())

def make_encoder(pretrained):
    if pretrained:
        checkpoint=torch.load(CHEMELEON_PATH,map_location='cpu',weights_only=True)
        assert 'hyper_parameters' in checkpoint and 'state_dict' in checkpoint
        mp=nn.BondMessagePassing(**checkpoint['hyper_parameters'])
        mp.load_state_dict(checkpoint['state_dict'])
    else:
        # Matched-capacity scratch control: initialize the SAME CheMeleon
        # architecture with fresh random weights, rather than Chemprop's
        # smaller default D-MPNN architecture.
        if CHEMELEON_PATH.exists():
            reference=torch.load(CHEMELEON_PATH,map_location='cpu',weights_only=True)
            mp=nn.BondMessagePassing(**reference['hyper_parameters'])
        else:
            mp=nn.BondMessagePassing()
            print('WARNING: no CheMeleon checkpoint; scratch capacity is unmatched.')
    return mp

## 4 · Frozen CheMeleon embeddings, computed in batches and cached

In [ ]:
def chemeleon_fingerprints(molecules,batch_size=64):
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    mp=make_encoder(pretrained=True)
    graph_model=models.MPNN(
        message_passing=mp,agg=nn.MeanAggregation(),
        predictor=nn.RegressionFFN(input_dim=mp.output_dim),batch_norm=False
    ).to(device)
    graph_model.eval()
    graph_featurizer=featurizers.SimpleMoleculeMolGraphFeaturizer()
    result=[]
    from chemprop.data import BatchMolGraph
    for start in range(0,len(molecules),batch_size):
        bmg=BatchMolGraph([graph_featurizer(m) for m in molecules[start:start+batch_size]])
        bmg.to(device=graph_model.device)
        with torch.inference_mode():
            emb=graph_model.fingerprint(bmg).detach().cpu().numpy()
        result.append(emb.astype(np.float32))
        if start % (batch_size*10)==0: print(f'  Embedded {min(start+batch_size,len(molecules))}/{len(molecules)}')
    del graph_model,mp
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return np.vstack(result)

EMB_PATH=CACHE_DIR/f'chemeleon_embeddings_{data_sha[:12]}.npy'
if RUN_FROZEN:
    if EMB_PATH.exists():
        emb=np.load(EMB_PATH)
        assert len(emb)==len(clean),'Cached embedding count differs; remove stale cache.'
        print('Loaded cached embedding matrix:',emb.shape)
    else:
        emb=chemeleon_fingerprints(mols,batch_size=BATCH_SIZE)
        np.save(EMB_PATH,emb)
        print('Cached embeddings:',EMB_PATH,emb.shape)
    assert np.isfinite(emb).all()

## 5 · Fit a frozen-embedding **per-isoform** LightGBM and Ridge baseline

In [ ]:
def metrics_for_arrays(yt,yp,lo=None,hi=None):
    result={'n':int(len(yt)),'mae':float(mean_absolute_error(yt,yp)),
            'r2':float(r2_score(yt,yp)),
            'spearman':float(spearmanr(yt,yp).statistic),
            'kendall':float(kendalltau(yt,yp).statistic),
            'pred_sd_over_true_sd':float(np.std(yp)/np.std(yt)) if np.std(yt)>0 else np.nan}
    if lo is not None and hi is not None:
        iv=np.maximum(lo-yp,0)+np.maximum(yp-hi,0)
        iv[(~np.isfinite(lo))|(~np.isfinite(hi))|(lo>hi)]=np.nan
        result['mean_interval_excess']=float(np.nanmean(iv)) if np.isfinite(iv).any() else np.nan
    return result

all_predictions=[];all_metrics=[]
def store_predictions(method,fold,indices,pred_matrix):
    assert pred_matrix.shape==(len(indices),len(ISOFORMS))
    for j,iso in enumerate(ISOFORMS):
        mask=np.isfinite(Y[indices,j]); vi=indices[mask]
        if not len(vi):continue
        yt=Y[vi,j];yp=pred_matrix[mask,j]
        lo_col=f'{iso}_pIC50_direct_inhibition_conf_low'
        hi_col=f'{iso}_pIC50_direct_inhibition_conf_high'
        lo=pd.to_numeric(clean.iloc[vi][lo_col],errors='coerce').to_numpy(float) if lo_col in clean else np.full(len(vi),np.nan)
        hi=pd.to_numeric(clean.iloc[vi][hi_col],errors='coerce').to_numpy(float) if hi_col in clean else np.full(len(vi),np.nan)
        stats=metrics_for_arrays(yt,yp,lo,hi)
        all_metrics.append(dict(model=method,fold=fold,isoform=iso,**stats))
        for k,idx in enumerate(vi):
            all_predictions.append(dict(model=method,fold=fold,isoform=iso,
                canonical_smiles=clean.loc[idx,'canonical_smiles'],y_true=float(yt[k]),
                y_pred=float(yp[k]),ci_low=float(lo[k]),ci_high=float(hi[k])))
        print(f'{method:24s} fold={fold} {iso}: MAE={stats["mae"]:.3f} (n={stats["n"]})')

if RUN_FROZEN:
    for fold in FOLDS_TO_RUN:
        tr=np.flatnonzero(folds!=fold);te=np.flatnonzero(folds==fold)
        for method in ['Frozen_CheMeleon_LGBM','Frozen_CheMeleon_Ridge']:
            p=np.full((len(te),len(ISOFORMS)),np.nan,dtype=float)
            for j,iso in enumerate(ISOFORMS):
                labeled=tr[np.isfinite(Y[tr,j])]
                if len(labeled)<20:continue
                if method.endswith('LGBM'):
                    model=lgb.LGBMRegressor(n_estimators=300,learning_rate=0.04,
                        num_leaves=15,min_child_samples=15,reg_lambda=3,verbosity=-1,n_jobs=2,
                        random_state=SEED)
                else:
                    model=make_pipeline(StandardScaler(),Ridge(alpha=100.0))
                model.fit(emb[labeled],Y[labeled,j]);p[:,j]=model.predict(emb[te])
            store_predictions(method,fold,te,p)

## 6 · Fit four-output Chemprop models: from scratch vs pretrained CheMeleon

In [ ]:
def make_dp(index,ys):
    # Chemprop accepts an array of targets with NaN for missing labels; losses are masked.
    return data.MoleculeDatapoint.from_smi(clean.loc[index,'SMILES'],np.asarray(ys,dtype=np.float32))

def make_inner_split(outer_training,seed):
    # Inner split uses only outer-training compounds and groups by the same scaffolds.
    subgroups=clean.iloc[outer_training]['scaffold_group'].to_numpy()
    inner=GroupShuffleSplit(n_splits=1,test_size=INNER_VAL_FRACTION,random_state=seed)
    a,b=next(inner.split(outer_training,groups=subgroups))
    fit_idx=outer_training[a];dev_idx=outer_training[b]
    assert not (set(clean.iloc[fit_idx]['scaffold_group'])&set(clean.iloc[dev_idx]['scaffold_group']))
    assert not (set(fit_idx)&set(dev_idx))
    return fit_idx,dev_idx

def train_multitask(outer_fold,pretrained=False):
    outer_train=np.flatnonzero(folds!=outer_fold)
    outer_test=np.flatnonzero(folds==outer_fold)
    fit_idx,dev_idx=make_inner_split(outer_train,SEED+outer_fold)
    # Small scaffold-group splits can leave an isoform unmeasured in dev: model selection
    # uses available targets; do not use outer-test labels to fill gaps.
    if not np.isfinite(Y[fit_idx]).any() or not np.isfinite(Y[dev_idx]).any():
        raise ValueError('No observed training or inner-validation labels')
    feat=featurizers.SimpleMoleculeMolGraphFeaturizer()
    train_set=data.MoleculeDataset([make_dp(i,Y[i]) for i in fit_idx],feat)
    scaler=train_set.normalize_targets() # critical: fit ONLY on the inner training set
    dev_set=data.MoleculeDataset([make_dp(i,Y[i]) for i in dev_idx],feat)
    dev_set.normalize_targets(scaler)
    test_set=data.MoleculeDataset([data.MoleculeDatapoint.from_smi(clean.loc[i,'SMILES']) for i in outer_test],feat)
    train_loader=data.build_dataloader(train_set,batch_size=BATCH_SIZE,num_workers=0,shuffle=True)
    val_loader=data.build_dataloader(dev_set,batch_size=BATCH_SIZE,num_workers=0,shuffle=False)
    test_loader=data.build_dataloader(test_set,batch_size=BATCH_SIZE,num_workers=0,shuffle=False)
    mp=make_encoder(pretrained=pretrained)
    transform=nn.UnscaleTransform.from_standard_scaler(scaler)
    predictor=nn.RegressionFFN(n_tasks=len(ISOFORMS),input_dim=mp.output_dim,
                               output_transform=transform)
    graph_model=models.MPNN(mp,nn.MeanAggregation(),predictor,batch_norm=False,
                             metrics=[nn.metrics.MAE()])
    model_name='CheMeleon_finetuned' if pretrained else 'Chemprop_scratch'
    ckpt_dir=OUT/'checkpoints'/f'{model_name}_fold{outer_fold}'
    ckpt_dir.mkdir(parents=True,exist_ok=True)
    checkpoint=ModelCheckpoint(dirpath=ckpt_dir,filename='best',monitor='val_loss',
                               mode='min',save_top_k=1,save_last=False)
    stopping=EarlyStopping(monitor='val_loss',patience=4 if FAST_MODE else 7,mode='min')
    trainer=pl.Trainer(max_epochs=MAX_EPOCHS,
                       accelerator='gpu' if torch.cuda.is_available() else 'cpu',devices=1,
                       logger=False,enable_model_summary=False,enable_progress_bar=True,
                       callbacks=[checkpoint,stopping],gradient_clip_val=1.0,
                       deterministic=False,log_every_n_steps=10)
    trainer.fit(graph_model,train_loader,val_loader)
    assert checkpoint.best_model_path, 'No best checkpoint saved; check training logs.'
    with torch.inference_mode():
        preds=trainer.predict(graph_model,dataloaders=test_loader,ckpt_path='best')
    p=np.concatenate([np.asarray(batch) for batch in preds],axis=0)
    p=np.asarray(p,dtype=float).reshape(len(outer_test),len(ISOFORMS))
    print('Fitted',model_name,'fold',outer_fold,'train/inner-val/test:',len(fit_idx),len(dev_idx),len(outer_test))
    return outer_test,p,dict(fold=outer_fold,model=model_name,
         n_inner_train=len(fit_idx),n_inner_val=len(dev_idx),
         checkpoint=str(checkpoint.best_model_path),best_val_loss=float(checkpoint.best_model_score))

training_runs=[]
for fold in FOLDS_TO_RUN:
    if RUN_SCRATCH:
        idx,pred,meta=train_multitask(fold,pretrained=False)
        store_predictions('Chemprop_scratch',fold,idx,pred);training_runs.append(meta)
    if RUN_CHEMELEON_FINETUNE:
        idx,pred,meta=train_multitask(fold,pretrained=True)
        store_predictions('CheMeleon_finetuned',fold,idx,pred);training_runs.append(meta)

## 7 · Compare only the same compounds and folds, then save results

In [ ]:
results=pd.DataFrame(all_predictions);fold_metrics=pd.DataFrame(all_metrics)
assert len(results)>0,'No models enabled or no valid predictions.'
assert not results.duplicated(['model','fold','isoform','canonical_smiles']).any()
results.to_csv(OUT/'predictions.csv',index=False)
fold_metrics.to_csv(OUT/'fold_metrics.csv',index=False)
# Summaries are valid only over folds actually computed. Do not confuse 1-fold demo with full OOF.
summary=[]
for (model,iso),part in results.groupby(['model','isoform']):
    s=metrics_for_arrays(part.y_true.to_numpy(),part.y_pred.to_numpy(),
                         part.ci_low.to_numpy(),part.ci_high.to_numpy())
    summary.append(dict(model=model,isoform=iso,n_folds=part.fold.nunique(),**s))
summary=pd.DataFrame(summary)
summary.to_csv(OUT/'summary.csv',index=False)
display(summary.round(4))

fig,ax=plt.subplots(figsize=(12,5))
for model,subset in summary.groupby('model'):
    ax.plot(subset.isoform,subset.mae,marker='o',label=model)
ax.set(xlabel='CYP isoform',ylabel='MAE (pIC50)',title=f'CheMeleon comparison; folds {FOLDS_TO_RUN} (not leaderboard scores)')
ax.legend(fontsize=8);plt.tight_layout();plt.savefig(OUT/'01_model_comparison.png',dpi=150);plt.show()

if NOTEBOOK05_ZIP_PATH:
    with zipfile.ZipFile(NOTEBOOK05_ZIP_PATH) as z:
        paths=[x for x in z.namelist() if x.endswith('/oof_predictions.csv')]
        if paths:
            baseline=pd.read_csv(io.BytesIO(z.read(paths[0])))
            baseline=baseline.rename(columns={'y_pred':'baseline_pred','model':'baseline_model'})
            for col in ['canonical_smiles','fold','isoform']:
                assert col in baseline.columns
            joined=results.merge(baseline[['canonical_smiles','fold','isoform','baseline_model','baseline_pred']],
                on=['canonical_smiles','fold','isoform'],how='inner')
            joined.to_csv(OUT/'matched_notebook05_comparison.csv',index=False)
            print('Matched with Notebook 05:',len(joined),'prediction pairings.')

meta=dict(notebook='06_chemeleon_foundation_model.ipynb',data_sha256=data_sha,
    folds_run=FOLDS_TO_RUN,n_splits=N_SPLITS,complete_oof=len(FOLDS_TO_RUN)==N_SPLITS,
    seed=SEED,fast_mode=FAST_MODE,max_epochs=MAX_EPOCHS,batch_size=BATCH_SIZE,
    n_molecules=len(clean),python=platform.python_version(),torch=torch.__version__,
    chemprop=getattr(chemprop,'__version__','unknown'),rdkit=rdBase.rdkitVersion,
    models_run=sorted(results.model.unique().tolist()),train_runs=training_runs,
    caveats=['single outer fold in FAST_MODE is a smoke test','inner train only used for graph weights',
      'experiment interval excess is not the official ST-RAE'])
(OUT/'metadata.json').write_text(json.dumps(meta,indent=2))
archive=Path('cyp_notebook_06_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,f'{OUT.name}/{f.name}')
print('Saved results:',archive,round(archive.stat().st_size/1024,1),'KiB')
print('Model checkpoints and cached embeddings are intentionally NOT included in result ZIP.')

## 8 · Research interpretation and next steps

1. Compare **CheMeleon_finetuned vs Chemprop_scratch**. Both instantiate CheMeleon's checkpoint-defined message-passing hyperparameters, but only the pretrained model loads its learned weights. This controls model capacity while testing the effect of pretraining.
2. Compare **Frozen_CheMeleon_LGBM** and **Frozen_CheMeleon_Ridge** to assess whether the representation is useful without supervised encoder adaptation.
3. For publication-quality results, run all 5 folds, consider several network random seeds, keep a separate grouped holdout for selection, and examine label-efficiency curves.
4. Check source/checkpoint licenses and cite OpenADMET, Chemprop, CheMeleon, and CheMeleon Zenodo checkpoint when publishing.
5. The **multi-assay auxiliary-head experiment** is reserved for Notebook 07; this notebook uses *only four direct-inhibition tasks*.